# Population and Sample

The course turns from probability to **statistics**, and everything in statistics rests on two concepts: the **population** — the entire group you want to study — and the **sample** — the smaller subset you actually observe. You can't measure all 10,000 inhabitants of Statistopia, so you measure 100 of them and *estimate*. Doing this well requires the sample to be **random**, **independent** (sampling with replacement), and **identically distributed** — the famous *i.i.d.* conditions — and in machine learning, where every dataset is a sample, it additionally has to be *representative*. It is based on the first video of Lesson 5 (Sampling and Point Estimation).

## 1. Introduction

Welcome to the statistics half of the course — and to the island of **Statistopia**, where you've been hired as data scientist. First task: find the average height of the islanders.

"Easy peasy — ask everyone and divide by the count." Then you learn the island has **10,000 people**. Not so easy anymore. The realistic strategy: ask only *a few* people and **estimate** the average. That switch — from measuring everyone to measuring a subset — is the founding move of statistics, and in machine learning it is unavoidable: models are trained on samples because the entire universe of data is never available.

After completing this notebook you will be able to:

- Define **population** (size $N$) and **sample** (size $n$) and identify each in a scenario.
- Explain the three sampling requirements: random selection, independence (sampling **with replacement**), and identical distribution — *i.i.d.*
- Demonstrate what goes wrong with sorted, dependent, or unrepresentative samples.
- Connect these ideas to machine-learning datasets (every dataset is a sample; representativeness matters).

## 2. Theory

### The two definitions

- **Population:** the entire set of individuals or elements you want to study, sharing some common behavior. All humans (for heights); all people of Statistopia; all avocados sold in the US; all possible cat images.
- **Sample:** a subset of the population that you actually observe or measure, used to draw conclusions about the population as a whole.

Notation: **capital $N$** for the population size, **lowercase $n$** for the sample size. Statistopia: $N = 10{,}000$; a sample could be any $n$ from 1 to 9,999 — small enough to be manageable, large enough to be significant.

### How to sample — and how not to

Suppose (for simplicity) Statistopia has only $N = 10$ people and you want $n = 4$ for your study. Two proposals:

1. **Pick 4 people at random.** ✔ Correct — you always want random samples.
2. **Line everyone up from shortest to tallest and take the first 4.** ✘ Bad — you'd be taking the shortest people, and your average would come out too small.

### Independence: sample with replacement

Now run the experiment *again* with a second sample of 4. If the second sample must avoid the people already chosen, it **depends on the first one** — and that ruins the experiment. The fix: **people are allowed to be repeated** (sampling *with replacement*), so every draw starts fresh and the samples are **independent**.

### Identically distributed: same rule every time

Whatever rule you use to pick the first sample must be the same rule for every other sample. If one day you sample in a part of town where people are taller (or shorter), your samples no longer follow the population's distribution. Samples must be **independent and identically distributed — i.i.d.**

### Populations and samples in machine learning

- **Every dataset is a sample**, never the population — no matter how big. A cat-image classifier's training set is a sample from the infinite population of possible cat and non-cat images.
- The sample must be **representative**: its distribution should match the population's. If every training cat sits in grass, the model may learn "grass ⇒ cat" — calling a cow in a field a cat, and missing a cat on a couch. Variety in the sample is what representativeness looks like in practice.

## 3. Worked Examples

### Example 1: Statistopia

Population: the 10,000 inhabitants ($N = 10{,}000$). Sample: the $n = 100$ randomly chosen islanders whose heights you record. The sample average is your *estimate* of the population average.

### Example 2: the avocado toast trend

You study the price of avocados in the US after the avocado-toast trend. You cannot check every purchase, so you randomly pick four stores and record their avocado sales.

- **Population:** *all* avocados sold in the US.
- **Sample:** the avocados sold by the four selected stores.

### Example 3: sorted sampling gone wrong

Ten islanders with heights (in cm) $150, 155, 158, 160, 163, 167, 170, 174, 178, 185$ — population mean $166$. Taking the first four of the sorted line: mean $= \frac{150+155+158+160}{4} = 155.75$ — an underestimate by more than 10 cm. A random sample of 4 has no such systematic bias (it errs, but in no preferred direction).

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

# The population: heights of all N = 10,000 inhabitants of Statistopia
N = 10_000
population = rng.normal(166, 9, size=N)
print(f"population size N = {N}")
print(f"population mean (the number we pretend not to know): {population.mean():.2f} cm")

### A random sample estimates the population mean

In [ ]:
rng = np.random.default_rng(7)

n = 100
sample = rng.choice(population, size=n, replace=True)
print(f"sample size n = {n}")
print(f"sample mean:     {sample.mean():.2f} cm")
print(f"population mean: {population.mean():.2f} cm")
print("One hundred people stand in for ten thousand -- and land close.")

### Good sample vs. sorted sample

The video's ten-person island: random selection vs. taking the first four of the height-sorted line.

In [ ]:
rng = np.random.default_rng(3)

ten_people = np.array([150, 155, 158, 160, 163, 167, 170, 174, 178, 185])
print(f"true mean of the 10 islanders: {ten_people.mean():.2f} cm")

sorted_first4 = np.sort(ten_people)[:4]
print(f"\n'first 4 of the sorted line': {sorted_first4} -> mean {sorted_first4.mean():.2f}")

means = [rng.choice(ten_people, size=4, replace=True).mean() for _ in range(10_000)]
print(f"random samples of 4 (10,000 repeats): average of sample means = "
      f"{np.mean(means):.2f}")
print("\nThe sorted pick is biased low by construction; random sampling errs in")
print("both directions and centers on the truth.")

### Why sampling WITH replacement keeps samples independent

Without replacement, the second draw's distribution depends on what the first removed — with replacement, every draw faces the identical population.

In [ ]:
rng = np.random.default_rng(0)

# a tiny population makes the effect obvious: one giant among four others
tiny = np.array([160.0, 162.0, 164.0, 166.0, 200.0])

trials = 200_000
# WITHOUT replacement: probability the 2nd person drawn is the 200cm giant,
# given the 1st person drawn was NOT the giant
draws = np.array([rng.choice(tiny, size=2, replace=False) for _ in range(trials)])
given = draws[draws[:, 0] != 200.0]
p_without = np.mean(given[:, 1] == 200.0)

# WITH replacement: same conditional probability
draws_w = rng.choice(tiny, size=(trials, 2), replace=True)
given_w = draws_w[draws_w[:, 0] != 200.0]
p_with = np.mean(given_w[:, 1] == 200.0)

print(f"P(2nd draw is the giant | 1st was not):")
print(f"  without replacement: {p_without:.3f}   (1/4 = 0.250 -- the 1st draw changed the odds)")
print(f"  with replacement:    {p_with:.3f}   (1/5 = 0.200 -- unchanged: independent)")

### Identically distributed: don't sample the tall part of town

Statistopia has neighborhoods. Sampling only Northville (where people are taller) breaks the 'identical' in i.i.d. — the estimates are systematically wrong.

In [ ]:
rng = np.random.default_rng(5)

# the same population, tagged by neighborhood
northville = population + 6      # a tall part of town (pretend these live there)
southburg = population - 6

def sample_mean_from(town, n=100):
    return rng.choice(town, size=n, replace=True).mean()

print(f"population mean: {population.mean():.2f}")
print(f"sampling only Northville: {sample_mean_from(northville):.2f}  (biased high)")
print(f"sampling only Southburg:  {sample_mean_from(southburg):.2f}  (biased low)")
print(f"sampling the whole island: {sample_mean_from(population):.2f}  (on target)")

## 5. Visualizations

### Population vs. sample

The whole island's height distribution, with one random sample of 100 drawn from it.

In [ ]:
rng = np.random.default_rng(7)
sample = rng.choice(population, size=100, replace=True)

plt.figure(figsize=(10, 5))
plt.hist(population, bins=60, density=True, alpha=0.5, color="tab:blue",
         label=f"population (N = {N:,})")
plt.hist(sample, bins=20, density=True, alpha=0.5, color="tab:orange",
         label="one random sample (n = 100)")
plt.axvline(population.mean(), color="tab:blue", linestyle="--", linewidth=2,
            label=f"population mean {population.mean():.1f}")
plt.axvline(sample.mean(), color="tab:red", linestyle="-", linewidth=2,
            label=f"sample mean {sample.mean():.1f}")
plt.title("A sample is a small, rougher copy of the population")
plt.xlabel("height (cm)")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Bigger samples estimate better

Sample means for growing $n$: small samples wobble, large ones settle onto the population mean — the payoff for "large enough to be significant".

In [ ]:
rng = np.random.default_rng(11)

ns = [2, 5, 10, 30, 100, 300, 1000, 3000]
plt.figure(figsize=(10, 5))
for n in ns:
    estimates = [rng.choice(population, size=n, replace=True).mean()
                 for _ in range(200)]
    plt.scatter([n] * 200, estimates, s=6, alpha=0.25, color="tab:blue")
plt.axhline(population.mean(), color="tab:red", linestyle="--", linewidth=2,
            label=f"population mean {population.mean():.1f}")
plt.xscale("log")
plt.title("200 sample means at each sample size n")
plt.xlabel("sample size n (log scale)")
plt.ylabel("sample mean (cm)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Random vs. biased sampling, side by side

Three sampling rules, 2,000 repetitions each: only the i.i.d. rule centers on the truth.

In [ ]:
rng = np.random.default_rng(2)

def biased_shortest(pop, n):
    # 'line them up and take from the short end' (with some randomness)
    shortest_quarter = np.sort(pop)[: len(pop) // 4]
    return rng.choice(shortest_quarter, size=n, replace=True)

def biased_neighborhood(pop, n):
    return rng.choice(pop + 6, size=n, replace=True)     # tall part of town

rules = [("random i.i.d.", lambda: rng.choice(population, 100, replace=True)),
         ("shortest quarter", lambda: biased_shortest(population, 100)),
         ("tall neighborhood", lambda: biased_neighborhood(population, 100))]

plt.figure(figsize=(10, 5))
for (name, rule), color in zip(rules, ["tab:green", "tab:orange", "tab:purple"]):
    means = [rule().mean() for _ in range(2000)]
    plt.hist(means, bins=40, alpha=0.6, color=color, label=name)
plt.axvline(population.mean(), color="tab:red", linestyle="--", linewidth=2,
            label="population mean")
plt.title("Distributions of sample means under three sampling rules")
plt.xlabel("sample mean (cm)")
plt.ylabel("count")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The representativeness problem, cat-classifier style

A toy version of cats-in-grass: the population has two clusters, but the unrepresentative sample only saw one — any conclusion drawn from it misdescribes the population.

In [ ]:
rng = np.random.default_rng(9)

# population of 'images': feature = greenness of the background
cats = rng.normal(7, 1.2, 5000)          # cats photographed mostly in grass
not_cats = rng.normal(3, 1.2, 5000)      # everything else, less green

biased_cats = cats[cats > 7]             # the dataset only has grass-heavy cats

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(cats, bins=50, alpha=0.6, color="tab:green", density=True, label="cats")
axes[0].hist(not_cats, bins=50, alpha=0.6, color="tab:brown", density=True,
             label="not cats")
axes[0].set_title("population: what images really look like")
axes[1].hist(biased_cats, bins=25, alpha=0.6, color="tab:green", density=True,
             label="cats in the dataset")
axes[1].hist(not_cats, bins=50, alpha=0.6, color="tab:brown", density=True,
             label="not cats")
axes[1].set_title("unrepresentative sample: only very green cats")
for ax in axes:
    ax.set_xlabel("background greenness")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("density")
plt.suptitle("A model trained on the right panel learns 'green = cat'")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** For each scenario, name the population and the sample: (a) polling 1,200 voters about a national election; (b) testing 50 light bulbs from a factory's daily production; (c) a Netflix A/B test shown to 1% of users.

**Exercise 2.** Statistopia's census bureau proposes sampling every 100th person from the *alphabetically sorted* citizen registry. Is this random? Is it likely biased for measuring heights? What about for measuring something correlated with family origin?

### Intermediate

**Exercise 3.** Simulate the video's second complaint: draw two consecutive samples of 4 from the 10-person island **without** replacement across samples (the second sample must avoid the first four people). Show that the second sample's mean depends on the first sample's mean (compute their correlation over many repetitions), and that with replacement the correlation vanishes.

**Exercise 4.** How big should $n$ be? For the Statistopia population, find (by simulation) the smallest $n$ for which 95% of sample means land within ±1 cm of the population mean. Compare with ±2 cm.

### Advanced

**Exercise 5.** Sampling with vs. without replacement *within* one sample: for $n = 100$ out of $N = 10{,}000$, show the two sample means are nearly identical in distribution — and explain why replacement barely matters when $n \ll N$ (what fraction of draws would repeat?).

**Exercise 6.** Build a representativeness check: given the population and a sample, compare their quantiles (a QQ-plot of sample vs population — reusing Lesson 3's tool). Run it on a fair random sample and on the tall-neighborhood sample; which diagnostic pattern exposes the bias?

## 7. Solutions

In [ ]:
# Exercise 1: naming populations and samples
print("Exercise 1:")
print("(a) population: all eligible voters;      sample: the 1,200 polled")
print("(b) population: the day's entire bulb production; sample: the 50 tested")
print("(c) population: all Netflix users;        sample: the 1% shown the test")

In [ ]:
# Exercise 2: systematic sampling from an alphabetical registry
print("Exercise 2: it is SYSTEMATIC, not random -- but for heights it is likely")
print("fine: alphabetical position is essentially unrelated to height, so the")
print("rule behaves like a random one. For traits correlated with family origin")
print("(surnames cluster by origin!), alphabetical order IS informative and the")
print("every-100th rule can systematically over/under-sample groups -- biased.")

In [ ]:
# Exercise 3: dependence across samples without replacement
rng = np.random.default_rng(0)

ten_people = np.array([150, 155, 158, 160, 163, 167, 170, 174, 178, 185])
trials = 20_000

m1_no, m2_no, m1_yes, m2_yes = [], [], [], []
for _ in range(trials):
    perm = rng.permutation(10)
    m1_no.append(ten_people[perm[:4]].mean())      # first sample
    m2_no.append(ten_people[perm[4:8]].mean())     # second must avoid the first
    m1_yes.append(rng.choice(ten_people, 4, replace=True).mean())
    m2_yes.append(rng.choice(ten_people, 4, replace=True).mean())

print(f"Exercise 3: corr(sample-1 mean, sample-2 mean)")
print(f"  without replacement across samples: {np.corrcoef(m1_no, m2_no)[0, 1]:+.3f}")
print(f"  with replacement (fresh each time): {np.corrcoef(m1_yes, m2_yes)[0, 1]:+.3f}")
print("Avoiding the first sample's people forces the second mean to move the")
print("opposite way (a tall first sample leaves short people behind) --")
print("negative correlation = dependence. Replacement erases it.")

In [ ]:
# Exercise 4: how big must n be?
rng = np.random.default_rng(1)

true_mean = population.mean()
print("Exercise 4:")
for target in [1.0, 2.0]:
    for n in [10, 30, 50, 100, 200, 300, 400, 600, 1000]:
        means = population[rng.integers(0, N, size=(3000, n))].mean(axis=1)
        frac = np.mean(np.abs(means - true_mean) <= target)
        if frac >= 0.95:
            print(f"  within +-{target:.0f} cm 95% of the time: n ~ {n} "
                  f"(achieved {frac:.3f})")
            break
print("Tighter targets need much larger samples -- accuracy is bought with n.")

In [ ]:
# Exercise 5: replacement barely matters when n << N
rng = np.random.default_rng(2)

with_r = [rng.choice(population, 100, replace=True).mean() for _ in range(4000)]
without_r = [rng.choice(population, 100, replace=False).mean() for _ in range(4000)]
print(f"Exercise 5: sample-mean std with replacement:    {np.std(with_r):.3f}")
print(f"            sample-mean std without replacement: {np.std(without_r):.3f}")
expected_repeats = 100 * 99 / 2 / N
print(f"Expected repeated-person pairs in one sample: ~{expected_repeats:.2f}")
print("With n/N = 1%, a with-replacement sample almost never repeats anyone, so")
print("the two schemes are practically identical -- the i.i.d. idealization is")
print("harmless for small sampling fractions.")

In [ ]:
# Exercise 6: a QQ-based representativeness check
rng = np.random.default_rng(3)

fair = rng.choice(population, 300, replace=True)
tall_town = rng.choice(population + 6, 300, replace=True)

qs = np.linspace(0.02, 0.98, 49)
pop_q = np.quantile(population, qs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, s, name in [(axes[0], fair, "fair random sample"),
                    (axes[1], tall_town, "tall-neighborhood sample")]:
    ax.plot(pop_q, pop_q, color="tab:orange", linewidth=2, label="y = x")
    ax.plot(pop_q, np.quantile(s, qs), "o", markersize=4, label="sample quantiles")
    ax.set_title(name)
    ax.set_xlabel("population quantiles")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("sample quantiles")
plt.suptitle("Exercise 6: quantile-vs-quantile representativeness check")
plt.tight_layout()
plt.show()

print("Exercise 6: the fair sample hugs y = x; the biased one floats a constant")
print("~6 cm ABOVE the line at every quantile -- a parallel offset is the")
print("signature of a location-shifted (non-identically-distributed) sample.")

## 8. Conclusion

- **Population** = the entire set you want to study (size $N$); **sample** = the subset you actually measure (size $n$), used to draw conclusions about the whole. Statistopia: $N = 10{,}000$, $n = 100$; avocados: all US avocado sales vs. four stores.
- Good samples are **i.i.d.**: chosen at **random** (never "the first four of the sorted line"), **independent** — sample *with replacement* so no sample depends on a previous one — and **identically distributed** — the same selection rule every time (no tall neighborhoods).
- Biased rules don't average away: sorted or one-neighborhood sampling misses the truth *systematically*, no matter how often repeated; random sampling errs without preference and centers on the target.
- When $n \ll N$, with- and without-replacement sampling are practically identical — the i.i.d. idealization is harmless.
- In machine learning **every dataset is a sample**, and it must be **representative** (sample distribution ≈ population distribution) — the cats-in-grass failure is a representativeness failure.

Next: what to *compute* from a sample — sample mean, sample proportion, and sample variance as **point estimates** of population quantities.